In [ ]:
"""
Model-based Adaptation to Expanding Action Spaces in Reinforcement Learning
Fixed version with MLP-based transition model for Highway environment

Fixes applied:
1. Base model now trains with BASE_ACTIONS (2 actions) instead of FULL_ACTION_LIST
2. Fixed action index mapping when expanding from base to full action space
3. Fixed action index/value consistency in act() and enhanced_epsilon_greedy()
4. Fixed transition model training frequency (now based on steps, not episodes)
5. Added memory size limit to DQN_MLP_Agent
6. Fixed epsilon decay in training loop
7. Removed duplicate imports
"""

import sys
import gymnasium as gym
import highway_env
import numpy as np
import matplotlib.pyplot as plt
from collections import deque

if not hasattr(np, 'bool8'):
    np.bool8 = np.bool_

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F


# =============================================================================
# DQN Network Architecture
# =============================================================================

class DQNNetwork(nn.Module):
    """Deep Q-Network with 3 hidden layers"""
    def __init__(self, state_size, action_size):
        super(DQNNetwork, self).__init__()
        self.fc1 = nn.Linear(state_size, 32)
        self.fc2 = nn.Linear(32, 32)
        self.fc3 = nn.Linear(32, 16)
        self.fc4 = nn.Linear(16, action_size)
    
    def forward(self, x):
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = F.relu(self.fc3(x))
        x = self.fc4(x)
        return x


# =============================================================================
# Base DQN Agent (for pre-training with limited actions)
# =============================================================================

class DQNAgent:
    """DQN Agent for training with a fixed action set"""
    def __init__(self, state_size, action_list):
        self.state_size = state_size
        self.action_list = action_list  # List of actual action values (e.g., [0, 2])
        self.action_to_idx = {action: idx for idx, action in enumerate(action_list)}
        self.memory = []
        self.max_memory_size = 5000
        self.gamma = 0.99
        self.epsilon = 1.0
        self.epsilon_min = 0.05
        self.epsilon_decay = 0.99
        self.learning_rate = 0.001
        
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.model = DQNNetwork(self.state_size, len(self.action_list)).to(self.device)
        self.optimizer = optim.Adam(self.model.parameters(), lr=self.learning_rate)
        self.criterion = nn.MSELoss()

    def remember(self, state, action_idx, reward, next_state, done):
        """Store experience in replay buffer"""
        self.memory.append((state, action_idx, reward, next_state, done))
        if len(self.memory) > self.max_memory_size:
            self.memory.pop(0)

    def act(self, state):
        """Select action using epsilon-greedy policy"""
        if np.random.rand() <= self.epsilon:
            action_idx = np.random.randint(len(self.action_list))
        else:
            state_tensor = torch.FloatTensor(state).to(self.device)
            self.model.eval()
            with torch.no_grad():
                q_values = self.model(state_tensor)
            self.model.train()
            action_idx = torch.argmax(q_values[0]).item()
        
        # Return both action value and index
        return self.action_list[action_idx], action_idx

    def replay(self, batch_size):
        """Experience replay for training"""
        minibatch = np.random.choice(len(self.memory), batch_size, replace=False)
        states = np.array([self.memory[i][0].flatten() for i in minibatch])
        actions = np.array([self.memory[i][1] for i in minibatch])
        rewards = np.array([self.memory[i][2] for i in minibatch])
        next_states = np.array([self.memory[i][3].flatten() for i in minibatch])
        dones = np.array([self.memory[i][4] for i in minibatch])

        states_tensor = torch.FloatTensor(states).to(self.device)
        actions_tensor = torch.LongTensor(actions).to(self.device)
        rewards_tensor = torch.FloatTensor(rewards).to(self.device)
        next_states_tensor = torch.FloatTensor(next_states).to(self.device)
        dones_tensor = torch.FloatTensor(dones).to(self.device)

        # Compute target Q-values
        self.model.eval()
        with torch.no_grad():
            next_q_values = self.model(next_states_tensor)
            max_next_q_values = torch.max(next_q_values, dim=1)[0]
        self.model.train()
        targets = rewards_tensor + self.gamma * max_next_q_values * (1 - dones_tensor)
        
        # Compute current Q-values
        current_q_values = self.model(states_tensor)
        current_q_values_for_actions = current_q_values.gather(1, actions_tensor.unsqueeze(1)).squeeze(1)
        
        # Update network
        loss = self.criterion(current_q_values_for_actions, targets)
        self.optimizer.zero_grad()
        loss.backward()
        self.optimizer.step()

        if self.epsilon > self.epsilon_min:
            self.epsilon *= self.epsilon_decay

    def train(self, env, episodes=200):
        """Train the agent"""
        BATCH_SIZE = 32
        MIN_MEMORY_SIZE = 200
        total_reward = []

        for e in range(episodes):
            state, _ = env.reset()
            state = np.reshape(state, [1, self.state_size])
            done = False
            reward_count = 0
            steps = 0

            while not done:
                action, action_idx = self.act(state)
                next_state, reward, terminated, truncated, _ = env.step(action)
                done = terminated or truncated
                reward_count += reward
                next_state = np.reshape(next_state, [1, self.state_size])
                self.remember(state, action_idx, reward, next_state, done)
                state = next_state
                steps += 1

                if len(self.memory) > MIN_MEMORY_SIZE:
                    self.replay(BATCH_SIZE)

            total_reward.append(reward_count)
            if (e + 1) % 10 == 0:
                print(f"Episode {e+1}/{episodes}, Reward: {reward_count:.2f}, Steps: {steps}, Epsilon: {self.epsilon:.3f}")

        env.close()
        return total_reward

    def load(self, name):
        try:
            state_dict = torch.load(name, map_location=self.device, weights_only=True)
        except TypeError:
            state_dict = torch.load(name, map_location=self.device)
        self.model.load_state_dict(state_dict)

    def save(self, name):
        torch.save(self.model.state_dict(), name)


# =============================================================================
# Transition Model Learner (MLP for predicting new action transitions)
# =============================================================================

class TransitionModelLearnerDQN(nn.Module):
    """MLP model to predict next state for newly introduced actions"""
    def __init__(self, state_dim, action_dim=1, hidden_dim=64, lr=0.001, buffer_size=10000):
        super(TransitionModelLearnerDQN, self).__init__()
        
        input_dim = state_dim + action_dim
        self.fc1 = nn.Linear(input_dim, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, hidden_dim)
        self.fc3 = nn.Linear(hidden_dim, state_dim)
        self.dropout = nn.Dropout(0.1)
        
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.to(self.device)
        self.optimizer = optim.Adam(self.parameters(), lr=lr)
        self.criterion = nn.MSELoss()
        
        self.buffer = deque(maxlen=buffer_size)
        self.min_buffer_size = 100
        
    def forward(self, x):
        x = F.relu(self.fc1(x))
        x = self.dropout(x)
        x = F.relu(self.fc2(x))
        x = self.dropout(x)
        x = self.fc3(x)
        return x
        
    def add_experience(self, state, next_state, action, actions_list):
        """Add transition experience for new actions"""
        min_action = min(actions_list)
        max_action = max(actions_list)
        # Normalize action to [0, 1]
        action_features = np.array([(action - min_action) / (max_action - min_action + 1e-8)])
        self.buffer.append((state, action_features, next_state))
    
    def can_predict(self):
        return len(self.buffer) >= self.min_buffer_size
    
    def train_model(self, batch_size=32, epochs=10):
        """Train the transition model on collected experiences"""
        if len(self.buffer) < self.min_buffer_size:
            return
            
        sample_size = min(len(self.buffer), 1000)
        samples = list(self.buffer)[-sample_size:]
        
        states = np.array([s[0].flatten() for s in samples])
        actions = np.array([s[1] for s in samples])
        next_states = np.array([s[2].flatten() for s in samples])
        
        states = torch.FloatTensor(states).to(self.device)
        actions = torch.FloatTensor(actions).to(self.device)
        next_states = torch.FloatTensor(next_states).to(self.device)
        
        self.train()
        for _ in range(epochs):
            indices = torch.randperm(len(states))
            for i in range(0, len(states), batch_size):
                batch_indices = indices[i:i+batch_size]
                batch_states = states[batch_indices]
                batch_actions = actions[batch_indices]
                batch_next_states = next_states[batch_indices]
                
                if batch_actions.dim() == 1:
                    batch_actions = batch_actions.unsqueeze(1)
                
                batch_input = torch.cat([batch_states, batch_actions], dim=1)
                predicted_next_states = self(batch_input)
                loss = self.criterion(predicted_next_states, batch_next_states)
                
                self.optimizer.zero_grad()
                loss.backward()
                self.optimizer.step()
    
    def predict_next_state(self, state, state_size, action, actions_list):
        """Predict next state for a given state-action pair"""
        self.eval()
        with torch.no_grad():
            min_action = min(actions_list)
            max_action = max(actions_list)
            action_features = np.array([(action - min_action) / (max_action - min_action + 1e-8)])
            state_flat = state.flatten()
            input_features = np.concatenate([state_flat, action_features])
            input_features = np.reshape(input_features, [1, state_size + 1])
            input_tensor = torch.FloatTensor(input_features).to(self.device)
            
            predicted = self(input_tensor)
            return predicted.cpu().numpy()


# =============================================================================
# DQN Agent with MLP-based Transition Model (for action space expansion)
# =============================================================================

class DQN_MLP_Agent:
    """DQN Agent that uses MLP transition model for adapting to new actions"""
    def __init__(self, state_size, action_size, fine_tune_model=None):
        self.state_size = state_size
        self.action_size = action_size  # Total number of actions (5)
        self.memory = []
        self.max_memory_size = 5000  # FIX #4: Added memory limit
        self.gamma = 0.9
        self.epsilon = 1.0
        self.epsilon_min = 0.01
        self.epsilon_decay = 0.99
        self.learning_rate = 0.001
        
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        
        if fine_tune_model is not None:
            self.model = fine_tune_model.to(self.device)
        else:
            self.model = DQNNetwork(self.state_size, self.action_size).to(self.device)
        
        self.optimizer = optim.Adam(self.model.parameters(), lr=self.learning_rate)
        self.criterion = nn.MSELoss()
        
        self.transition_learner = TransitionModelLearnerDQN(state_dim=state_size, action_dim=1)
        self.rng = np.random.default_rng(123)
        
        # Statistics
        self.reuse = 0
        self.reject = 0

    def remember(self, state, action, reward, next_state, done):
        """Store experience in replay buffer with size limit"""
        self.memory.append((state, action, reward, next_state, done))
        # FIX #4: Enforce memory limit
        if len(self.memory) > self.max_memory_size:
            self.memory.pop(0)

    def act(self, state, action_list):
        """Select action using epsilon-greedy policy
        
        FIX #2: Consistent return of action VALUE (not index)
        """
        if np.random.rand() <= self.epsilon:
            return np.random.choice(action_list)
        
        state_tensor = torch.FloatTensor(state).to(self.device)
        self.model.eval()
        with torch.no_grad():
            q_values = self.model(state_tensor)
        self.model.train()
        
        # Get Q-values only for actions in action_list
        valid_q_values = [(action, q_values[0, action].item()) for action in action_list]
        best_action = max(valid_q_values, key=lambda x: x[1])[0]
        return best_action
    
    def enhanced_epsilon_greedy(self, state, base_actions, expand_actions, encourage_new_action=False):
        """Enhanced epsilon-greedy with option to encourage new action exploration
        
        FIX #2 & #5: Use self.epsilon (decaying) and return consistent action values
        """
        all_actions = base_actions + expand_actions
        
        # Encourage exploration of new actions early in training
        if encourage_new_action and self.rng.random() < 0.3:
            return np.random.choice(expand_actions)
        
        # Standard epsilon-greedy
        if self.rng.random() < self.epsilon:
            return np.random.choice(all_actions)
        
        # Greedy selection
        state_tensor = torch.FloatTensor(state).to(self.device)
        self.model.eval()
        with torch.no_grad():
            q_values = self.model(state_tensor)
        self.model.train()
        
        # Select best action from all available actions
        valid_q_values = [(action, q_values[0, action].item()) for action in all_actions]
        best_action = max(valid_q_values, key=lambda x: x[1])[0]
        return best_action
    
    def replay(self, batch_size):
        """Experience replay for training"""
        minibatch = np.random.choice(len(self.memory), batch_size, replace=False)
        states = np.array([self.memory[i][0].flatten() for i in minibatch])
        actions = np.array([self.memory[i][1] for i in minibatch])
        rewards = np.array([self.memory[i][2] for i in minibatch])
        next_states = np.array([self.memory[i][3].flatten() for i in minibatch])
        dones = np.array([self.memory[i][4] for i in minibatch])

        states_tensor = torch.FloatTensor(states).to(self.device)
        actions_tensor = torch.LongTensor(actions).to(self.device)
        rewards_tensor = torch.FloatTensor(rewards).to(self.device)
        next_states_tensor = torch.FloatTensor(next_states).to(self.device)
        dones_tensor = torch.FloatTensor(dones).to(self.device)

        self.model.eval()
        with torch.no_grad():
            next_q_values = self.model(next_states_tensor)
            max_next_q_values = torch.max(next_q_values, dim=1)[0]
        
        targets = rewards_tensor + self.gamma * max_next_q_values * (1 - dones_tensor)
        
        self.model.train()
        current_q_values = self.model(states_tensor)
        current_q_values_for_actions = current_q_values.gather(1, actions_tensor.unsqueeze(1)).squeeze(1)
        
        loss = self.criterion(current_q_values_for_actions, targets)
        
        self.optimizer.zero_grad()
        loss.backward()
        self.optimizer.step()

        # FIX #5: Decay epsilon here (it will be used in next step)
        if self.epsilon > self.epsilon_min:
            self.epsilon *= self.epsilon_decay

    def train(self, env, base_actions, expand_actions, episodes=200):
        """Train agent with action filtering for new actions
        
        FIX #3: Train transition model based on steps, not episodes
        """
        MODEL_TRAIN_INTERVAL = 100  # Train transition model every N steps
        BATCH_SIZE = 32
        total_rewards_log = []
        self.reuse = 0
        self.reject = 0
        total_steps = 0
        
        all_actions = base_actions + expand_actions
        
        for ep in range(episodes):
            state, _ = env.reset()
            state = np.reshape(state, [1, self.state_size])
            total_reward = 0
            done = False
            
            while not done:
                # Select action with enhanced epsilon-greedy
                action = self.enhanced_epsilon_greedy(
                    state, base_actions, expand_actions, 
                    encourage_new_action=(ep < 30)
                )
                
                # Action filtering for new actions (Equation 7 from paper)
                if action in expand_actions and self.transition_learner.can_predict():
                    # Predict next state using transition model
                    snext_model = self.transition_learner.predict_next_state(
                        state, self.state_size, action, expand_actions
                    )
                    
                    # Compare state values (V*(s'_model) vs V*(s))
                    snext_model_tensor = torch.FloatTensor(snext_model).to(self.device)
                    state_tensor = torch.FloatTensor(state).to(self.device)
                    
                    self.model.eval()
                    with torch.no_grad():
                        q_next_model = self.model(snext_model_tensor)
                        q_current = self.model(state_tensor)
                    self.model.train()
                    
                    # Accept new action only if it improves state value
                    if torch.max(q_next_model).item() > torch.max(q_current).item():
                        self.reuse += 1
                    else:
                        self.reject += 1
                        # Fallback to full action space (not just base actions)
                        all_actions = base_actions + expand_actions
                        action = self.act(state, all_actions)
                
                # Execute action
                next_state, reward, terminated, truncated, _ = env.step(action)
                done = terminated or truncated
                total_reward += reward
                next_state = np.reshape(next_state, [1, self.state_size])
                
                # Store experience for new actions in transition model
                if action in expand_actions:
                    self.transition_learner.add_experience(state, next_state, action, expand_actions)
                
                # Store in DQN replay buffer (action is already the index for highway env)
                self.remember(state, action, reward, next_state, done)
                state = next_state
                total_steps += 1
                
                # FIX #3: Train transition model based on steps
                if total_steps % MODEL_TRAIN_INTERVAL == 0 and self.transition_learner.can_predict():
                    self.transition_learner.train_model(batch_size=32, epochs=2)
                
                # Train DQN
                if len(self.memory) > BATCH_SIZE:
                    self.replay(BATCH_SIZE)
            
            total_rewards_log.append(total_reward)
            
            if (ep + 1) % 10 == 0:
                print(f"Episode {ep+1}/{episodes}, Reward: {total_reward:.2f}, "
                      f"Epsilon: {self.epsilon:.3f}, Reuse: {self.reuse}, Reject: {self.reject}")
        
        env.close()
        return total_rewards_log

    def load(self, name):
        self.model.load_state_dict(torch.load(name, map_location=self.device))

    def save(self, name):
        torch.save(self.model.state_dict(), name)


# =============================================================================
# Experiment Functions
# =============================================================================

def expand_model_weights(base_state_dict, base_actions, expand_actions, full_action_list, state_size, device):
    """
    Expand model weights from base action space to full action space
    
    FIX #1 & #2: Proper action index mapping
    """
    # Create new model with full action space
    full_model = DQNNetwork(state_size, len(full_action_list)).to(device)
    
    # Copy all layers except the last one (fc4)
    filtered_state_dict = {k: v for k, v in base_state_dict.items() if not k.startswith("fc4.")}
    full_model.load_state_dict(filtered_state_dict, strict=False)
    
    # Get base model's last layer weights
    base_w = base_state_dict["fc4.weight"].to(device)  # Shape: [num_base_actions, 16]
    base_b = base_state_dict["fc4.bias"].to(device)    # Shape: [num_base_actions]
    
    # Create mapping from base actions to their indices in base model
    base_action_to_idx = {action: idx for idx, action in enumerate(base_actions)}
    
    # Initialize new weights
    new_w = full_model.fc4.weight.detach().clone()
    new_b = full_model.fc4.bias.detach().clone()
    
    # Copy weights for base actions to their correct positions
    for action in base_actions:
        base_idx = base_action_to_idx[action]
        new_w[action] = base_w[base_idx]  # Map to correct action index
        new_b[action] = base_b[base_idx]
    
    # Initialize new actions with random base action weights (optimistic initialization)
    for action in expand_actions:
        src_action = np.random.choice(base_actions)
        src_idx = base_action_to_idx[src_action]
        new_w[action] = base_w[src_idx]
        new_b[action] = base_b[src_idx]
    
    # Apply expanded weights
    with torch.no_grad():
        full_model.fc4.weight.copy_(new_w)
        full_model.fc4.bias.copy_(new_b)
    
    return full_model


def run_multiple_experiments(base_actions, expand_actions, full_action_list, state_size,
                             n_runs=5, base_seed=123, episodes=200, model_path="highway_dqn_model.pth"):
    """Run multiple training experiments with different seeds and return statistics"""
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    
    all_returns_naive = []
    all_returns_ours = []
    all_reuse_counts = []
    all_reject_counts = []
    
    for run in range(n_runs):
        current_seed = base_seed + run * 42
        np.random.seed(current_seed)
        torch.manual_seed(current_seed)
        
        print(f"\n{'='*60}")
        print(f"Running experiment {run + 1}/{n_runs} (seed={current_seed})")
        print(f"{'='*60}")
        
        # 1. Naive baseline: Train from scratch with full action space
        print(f"\n[Run {run+1}] Training NAIVE agent (5 actions from scratch)...")
        env_naive = gym.make("highway-v0", render_mode="rgb_array")
        naive_agent = DQNAgent(state_size, full_action_list)
        naive_returns = naive_agent.train(env_naive, episodes=episodes)
        all_returns_naive.append(naive_returns)
        
        # 2. Our method: Load pre-trained base model and expand
        print(f"\n[Run {run+1}] Training OUR agent (expanding 2 -> 5 actions)...")
        env_ours = gym.make("highway-v0", render_mode="rgb_array")
        
        # Load base model weights
        try:
            base_state_dict = torch.load(model_path, map_location=device, weights_only=True)
        except TypeError:
            base_state_dict = torch.load(model_path, map_location=device)
        
        # Expand model weights
        expanded_model = expand_model_weights(
            base_state_dict, base_actions, expand_actions, 
            full_action_list, state_size, device
        )
        
        # Train with our method
        our_agent = DQN_MLP_Agent(state_size, len(full_action_list), fine_tune_model=expanded_model)
        our_returns = our_agent.train(env_ours, base_actions, expand_actions, episodes=episodes)
        all_returns_ours.append(our_returns)
        
        all_reuse_counts.append(our_agent.reuse)
        all_reject_counts.append(our_agent.reject)
        
        print(f"\n[Run {run+1}] Summary:")
        print(f"  Naive agent final reward: {naive_returns[-1]:.2f}")
        print(f"  Our agent final reward: {our_returns[-1]:.2f}")
        print(f"  Reuse count: {our_agent.reuse}")
        print(f"  Reject count: {our_agent.reject}")
    
    # Calculate statistics
    all_returns_naive = np.array(all_returns_naive)
    all_returns_ours = np.array(all_returns_ours)
    
    return {
        'returns': {
            'naive': {'mean': np.mean(all_returns_naive, axis=0), 
                      'std': np.std(all_returns_naive, axis=0)},
            'ours': {'mean': np.mean(all_returns_ours, axis=0), 
                     'std': np.std(all_returns_ours, axis=0)},
        },
        'info': {
            'reuse_counts': all_reuse_counts,
            'reject_counts': all_reject_counts
        }
    }


def plot_results(stats, window=10, figsize=(14, 6), save_path=None):
    """Plot results from multiple experiments with shaded error bars"""
    
    def moving_avg(data, w):
        return np.convolve(data, np.ones(w)/w, mode='valid')
    
    naive_mean = stats['returns']['naive']['mean']
    naive_std = stats['returns']['naive']['std']
    ours_mean = stats['returns']['ours']['mean']
    ours_std = stats['returns']['ours']['std']
    
    # Apply smoothing
    naive_mean_smooth = moving_avg(naive_mean, window)
    naive_std_smooth = moving_avg(naive_std, window)
    ours_mean_smooth = moving_avg(ours_mean, window)
    ours_std_smooth = moving_avg(ours_std, window)
    
    x = np.arange(len(naive_mean_smooth))
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=figsize)
    
    # Returns plot
    ax1.plot(x, naive_mean_smooth, color='blue', label='Naive Training (5 actions)', linewidth=2)
    ax1.fill_between(x, naive_mean_smooth - naive_std_smooth, naive_mean_smooth + naive_std_smooth,
                     color='blue', alpha=0.2)
    
    ax1.plot(x, ours_mean_smooth, color='green', label='Our Method (MLP-based)', linewidth=2)
    ax1.fill_between(x, ours_mean_smooth - ours_std_smooth, ours_mean_smooth + ours_std_smooth,
                     color='green', alpha=0.2)
    
    ax1.set_xlabel('Episode')
    ax1.set_ylabel('Total Reward')
    ax1.set_title('Training Rewards: Naive vs Our Method')
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    
    # Reuse/Reject statistics
    reuse_counts = stats['info']['reuse_counts']
    reject_counts = stats['info']['reject_counts']
    
    x_runs = np.arange(1, len(reuse_counts) + 1)
    width = 0.35
    
    ax2.bar(x_runs - width/2, reuse_counts, width, color='green', label='Reuse (Accept)', alpha=0.8)
    ax2.bar(x_runs + width/2, reject_counts, width, color='red', label='Reject (Fallback)', alpha=0.8)
    
    ax2.set_xlabel('Run')
    ax2.set_ylabel('Count')
    ax2.set_title('Action Filtering Statistics per Run')
    ax2.legend()
    ax2.grid(True, axis='y', alpha=0.3)
    
    plt.tight_layout()
    
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
        print(f"Figure saved to {save_path}")
    
    plt.show()
    
    # Print summary statistics
    print("\n" + "="*60)
    print("SUMMARY STATISTICS")
    print("="*60)
    print(f"\nNaive Training (5 actions from scratch):")
    print(f"  Final reward (mean ± std): {naive_mean[-1]:.2f} ± {naive_std[-1]:.2f}")
    print(f"  Max reward achieved: {np.max(naive_mean):.2f}")
    
    print(f"\nOur Method (MLP-based adaptation):")
    print(f"  Final reward (mean ± std): {ours_mean[-1]:.2f} ± {ours_std[-1]:.2f}")
    print(f"  Max reward achieved: {np.max(ours_mean):.2f}")
    
    print(f"\nAction Filtering Statistics:")
    print(f"  Average reuse per run: {np.mean(reuse_counts):.2f} ± {np.std(reuse_counts):.2f}")
    print(f"  Average reject per run: {np.mean(reject_counts):.2f} ± {np.std(reject_counts):.2f}")
    if np.mean(reuse_counts) + np.mean(reject_counts) > 0:
        print(f"  Reuse ratio: {np.mean(reuse_counts) / (np.mean(reuse_counts) + np.mean(reject_counts)) * 100:.1f}%")


# =============================================================================
# Main Execution
# =============================================================================

if __name__ == "__main__":
    # Configuration
    EPISODES_PRETRAIN = 100  # Episodes for pre-training base model
    EPISODES_EXPERIMENT = 100  # Episodes for each experiment run
    N_RUNS = 3  # Number of experiment runs for statistics
    
    # Highway environment action space:
    # {'LANE_LEFT': 0, 'IDLE': 1, 'LANE_RIGHT': 2, 'FASTER': 3, 'SLOWER': 4}
    BASE_ACTIONS = [0, 2]  # LANE_LEFT, LANE_RIGHT (pre-trained actions)
    EXPAND_ACTIONS = [1, 3, 4]  # IDLE, FASTER, SLOWER (new actions)
    FULL_ACTION_LIST = [0, 1, 2, 3, 4]  # All actions
    
    # Setup environment
    env = gym.make("highway-v0", render_mode="rgb_array")
    obs_space = env.observation_space
    state_size = np.prod(obs_space.shape) if hasattr(obs_space, 'shape') else obs_space.n
    
    print(f"State size: {state_size}")
    print(f"Base actions: {BASE_ACTIONS}")
    print(f"Expand actions: {EXPAND_ACTIONS}")
    print(f"Full action list: {FULL_ACTION_LIST}")
    
    # ==========================================================================
    # Step 1: Pre-train base model with BASE_ACTIONS only
    # ==========================================================================
    print("\n" + "="*60)
    print("PHASE 1: Pre-training base model with 2 actions")
    print("="*60)
    
    # FIX #1: Use BASE_ACTIONS instead of FULL_ACTION_LIST
    base_agent = DQNAgent(state_size, BASE_ACTIONS)
    base_total_reward_log = base_agent.train(env, episodes=EPISODES_PRETRAIN)
    base_agent.save("highway_dqn_model.pth")
    print("Base model saved to highway_dqn_model.pth")
    
    # ==========================================================================
    # Step 2: Run experiments comparing naive vs our method
    # ==========================================================================
    print("\n" + "="*60)
    print("PHASE 2: Running comparison experiments")
    print("="*60)
    
    stats = run_multiple_experiments(
        base_actions=BASE_ACTIONS,
        expand_actions=EXPAND_ACTIONS,
        full_action_list=FULL_ACTION_LIST,
        state_size=state_size,
        n_runs=N_RUNS,
        base_seed=123,
        episodes=EPISODES_EXPERIMENT,
        model_path="highway_dqn_model.pth"
    )
    
    # ==========================================================================
    # Step 3: Plot and save results
    # ==========================================================================
    print("\n" + "="*60)
    print("PHASE 3: Plotting results")
    print("="*60)
    
    plot_results(stats, window=10, save_path="experiment_results.png")

/var/folders/7y/tb8vlfv937523rgqdlf0zjy40000gn/T/ipykernel_6295/3141754529.py:22: DeprecationWarning: `np.bool8` is a deprecated alias for `np.bool_`.  (Deprecated NumPy 1.24)
  if not hasattr(np, 'bool8'):


State size: 25
Base actions: [0, 2]
Expand actions: [1, 3, 4]
Full action list: [0, 1, 2, 3, 4]

PHASE 1: Pre-training base model with 2 actions
Episode 10/100, Reward: 11.86, Steps: 15, Epsilon: 1.000
Episode 20/100, Reward: 4.90, Steps: 7, Epsilon: 0.878
Episode 30/100, Reward: 2.48, Steps: 4, Epsilon: 0.263
Episode 40/100, Reward: 12.08, Steps: 15, Epsilon: 0.122
Episode 50/100, Reward: 25.17, Steps: 30, Epsilon: 0.050
Episode 60/100, Reward: 5.24, Steps: 7, Epsilon: 0.050
Episode 70/100, Reward: 5.27, Steps: 7, Epsilon: 0.050
Episode 80/100, Reward: 23.39, Steps: 28, Epsilon: 0.050
Episode 90/100, Reward: 13.91, Steps: 17, Epsilon: 0.050
Episode 100/100, Reward: 4.38, Steps: 6, Epsilon: 0.050
Base model saved to highway_dqn_model.pth

PHASE 2: Running comparison experiments

Running experiment 1/3 (seed=123)

[Run 1] Training NAIVE agent (5 actions from scratch)...
Episode 10/100, Reward: 3.04, Steps: 4, Epsilon: 1.000
Episode 20/100, Reward: 7.16, Steps: 8, Epsilon: 0.257
Episode 